# Installs and imports

In [ ]:
#Basic imports
import os
import random
import numpy as np
import copy as cp
import sys
from tqdm import tqdm
import nltk
import math
import spacy

#Dataset imports
from transformers import DataCollatorWithPadding
import pandas as pd 
import datasets
import re
import json
import unicodedata

#Model fine-tuning imports
import torch
from torch.utils.data import DataLoader 
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
import time
from timeit import default_timer as timer
import datetime


from IPython.display import display, HTML
from matplotlib import pyplot as plt
import matplotlib

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

from sklearn.model_selection import learning_curve
from sklearn.svm import SVC
from sklearn.datasets import load_digits

from sklearn import preprocessing
from BERT_explainability.modules.BERT.BertForSequenceClassification import CamembertForSequenceClassification
from BERT_explainability.modules.BERT.ExplanationGenerator import Generator


print(torch.__version__)

# Your GPU architecture should be listed in the command below.
# If not, download another cuda version or use CPU
print(torch.cuda.get_arch_list())
torch.cuda.is_available()

In [ ]:
# => Libs python à installer pour pouvoir lancer le code en local et à mettre dans un requirements.txt

#!python3 -m pip install nltk
#!python3 -m pip install numpy
#!python3 -m pip install spacy
#!python3 -m pip install transformers
#!python3 -m pip install pandas
#!python3 -m pip install datasets
#!python3 -m pip install torch 
#!python3 -m pip install matplotlib
#!python3 -m pip install scikit-learn
#!python3 -m pip install openpyxl

In [ ]:
import subprocess as sp
import os
import psutil

def get_gpu_memory():
    command = "nvidia-smi --query-gpu=memory.free --format=csv"
    memory_free_info = sp.check_output(command.split()).decode('ascii').split('\n')[:-1][1:]
    memory_free_values = [int(x.split()[0]) for i, x in enumerate(memory_free_info)]
    return memory_free_values

def get_cpu_memory():
    return psutil.cpu_percent(1)


In [ ]:
def seed_worker(worker_id):
    worker_seed = seed
    numpy.random.seed(worker_seed)
    random.seed(worker_seed)
    
def set_seed(seed):
    """Set all seeds to make results reproducible (deterministic mode).
       When seed is None, disables deterministic mode.
    :param seed: an integer to your choosing
    """
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        np.random.seed(seed)
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
        
seed = 0
set_seed(seed)
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

## Model training

In [ ]:
import time 
import copy as cp
from torch.nn import Softmax


def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))


def train_model(model, train_set, val, device, epochs=2, lr = 2e-5, inplace=False):
    
    if not inplace:
        model = cp.deepcopy(model)
    
    optimizer = torch.optim.AdamW(model.parameters(),
                  lr = lr, 
                  eps = 1e-8
                )

    batches_per_epoch = len(train_set) // batch_size

    total_steps = int(batches_per_epoch * epochs)

    scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

    
    training_stats = []

    total_t0 = time.time()
    model.train()
    
    for epoch_i in range(0, epochs):

        # ========================================
        #               Training
        # ========================================

        print("")
        print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
        print('Training...')

        t0 = time.time()

        total_train_loss = 0
        all_input_ids = []
        for step, batch in enumerate(tqdm(train_set, total=len(train_set))):

            model.zero_grad()
            optimizer.zero_grad()
            
            cp_batch = cp.deepcopy(batch)
            b_input_ids = cp_batch['input_ids'].to(device)
            all_input_ids.append(b_input_ids.detach().cpu())
            b_input_mask = cp_batch['attention_mask'].to(device)
            b_labels = cp_batch['labels'].to(device)        

            loss, logits = model(b_input_ids, 
                                 token_type_ids=None, 
                                 attention_mask=b_input_mask, 
                                 labels=b_labels)[:2]

            total_train_loss += loss.item()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            scheduler.step()

        avg_train_loss = total_train_loss / len(train_set)            

        training_time = format_time(time.time() - t0)

        print("")
        print("  Average training loss: {0:.2f}".format(avg_train_loss))
        print("  Training epcoh took: {:}".format(training_time))

        # ========================================
        #               Validation
        # ========================================

        print("")
        print("Running Validation...")

        t0 = time.time()

        model.eval()

        total_eval_accuracy = 0
        total_eval_loss = 0
        nb_eval_steps = 0

        for step, batch in enumerate(tqdm(val, total=len(val))):

            cp_batch = cp.deepcopy(batch)
            b_input_ids = cp_batch['input_ids'].to(device)
            b_input_mask = cp_batch['attention_mask'].to(device)
            b_labels = cp_batch['labels'].to(device)

            with torch.no_grad():        

                loss, logits = model(b_input_ids, 
                                        token_type_ids=None, 
                                        attention_mask=b_input_mask,
                                        labels=b_labels)[:2]

            total_eval_loss += loss.item()

            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()

            total_eval_accuracy += flat_accuracy(logits, label_ids)


        avg_val_accuracy = total_eval_accuracy / len(val)
        print("  Accuracy: {0:.2f}".format(avg_val_accuracy))

        avg_val_loss = total_eval_loss / len(val)

        validation_time = format_time(time.time() - t0)

        print("  Validation Loss: {0:.2f}".format(avg_val_loss))
        print("  Validation took: {:}".format(validation_time))

        training_stats.append(
            {
                'epoch': epoch_i + 1,
                'Training Loss': avg_train_loss,
                'Valid. Loss': avg_val_loss,
                'Valid. Accur.': avg_val_accuracy,
                'Training Time': training_time,
                'Validation Time': validation_time
            }
        )

    print("")
    print("Training complete!")

    print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))
    
    for optimizer_metrics in optimizer.state.values():
        for metric_name, metric in optimizer_metrics.items():
            if torch.is_tensor(metric):
                optimizer_metrics[metric_name] = metric.cpu()
            
    return model, all_input_ids

def test_model(model, test, device, get_preds = False):
    print("")
    print("Running Test...")
    t0 = time.time()
    total_eval_accuracy = 0

    model.eval()

    predictions, all_preds, true_labels = [], [], []

    for step, batch in enumerate(tqdm(test, total = len(test))):

        cp_batch = cp.deepcopy(batch)
        b_input_ids = cp_batch['input_ids'].to(device)
        b_input_mask = cp_batch['attention_mask'].to(device)
        b_labels = cp_batch['labels'].to(device)

        with torch.no_grad():        
            outputs = model(b_input_ids, token_type_ids=None, 
                          attention_mask=b_input_mask)

        logits = outputs[0]
        
        soft = Softmax()
        preds = soft(logits).detach().cpu().numpy()
        predictions += [i for i in np.argmax(preds, axis=1)]
        all_preds += [i[1] for i in preds]
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        total_eval_accuracy += flat_accuracy(logits, label_ids)
        
        true_labels += [i for i in label_ids]

    avg_test_accuracy = np.mean([int(i == j) for i, j in zip(predictions, true_labels)])

    print("  Accuracy: {0:.4f}".format(avg_test_accuracy))
    return avg_test_accuracy
    if get_preds:
        return predictions, all_preds

In [ ]:
def normalize(vector, fullExpl=True, sample=False, s=20, topP=False, p=0.5, filter=False, visualize=False):
    neg = sum(vector) < 0
    if neg:
        vector = [-i for i in vector]
        
    min_val = min(vector)
    max_val = max(vector)
    vect_len = len(vector)
    assert max_val != min_val
    normalized = [(i-min_val)/(max_val-min_val) for i in vector]
    
    if fullExpl:
        normalized = [i/sum(normalized) for i in normalized] 
    
    if sample:
        s_norm = sorted([(pos, i) for pos, i in enumerate(normalized)], reverse = True, key = lambda a: a[1])
        sampled = sorted([(initPos, i) if pos < s else (initPos, 0) for pos, (initPos, i) in enumerate(s_norm)])
        normalized = [i for pos, i in sampled]
    if topP:
        s_norm = sorted([(pos, i) for pos, i in enumerate(normalized)], reverse = True, key = lambda a: a[1])
        pTot = 0
        vect_sum = sum([i[1] for i in s_norm])
        sampled = []
        for pos, (initPos, i) in enumerate(s_norm):
            if pTot <= p*vect_sum:
                pTot = sum(i[1] for i in s_norm[:pos])
                sampled.append((initPos, i))
            else:
                sampled.append((initPos, 0))
            
        sampled = sorted(sampled)
        normalized = [i for pos, i in sampled]
    if filter:
        normalized = [i if i>1/len(normalized) else 0 for i in normalized]
    if neg:
        normalized = [-i for i in normalized]

    if visualize:
        normalized = [i*vect_len for i in normalized]
    return normalized

In [ ]:
from Code_helpers.data_processing import convert_to_tok, retokenize, get_text_tokens
import gc

def explain_camembert(texts, tokenizer, model, explainer, device='cpu'):

    text_batch = texts
    dict_explanations = {}
    expls_bert = []
    y_pred_bert = []
    classifications = ["information", "opinion"]
    torch.set_grad_enabled(True)
    
    for nbr, itm in enumerate(tqdm(text_batch, total = len(text_batch))):  
        print(get_gpu_memory()[0])
        itm = get_text_tokens(itm, tokenizer, shorten=True)
        
        encoding = tokenize([itm], tokenizer)
        special_tokens = encoding['special_tokens']
        input_ids = torch.tensor(encoding['input_ids']).to(device)
        attention_mask = torch.tensor(encoding['attention_mask']).to(device)

        expl = explainer.generate_LRP(input_ids=input_ids, attention_mask=attention_mask, start_layer=1)[0]  
        expl.cpu()
        
        
        # get the model classification
        output = torch.nn.functional.softmax(model(input_ids=input_ids, attention_mask=attention_mask)[1], dim=-1).to("cpu")
        classification = output.argmax(dim=-1).item()
        y_pred_bert += [classification]
        class_name = classifications[classification]
        
        # if the classification is negative, higher explanation scores are more negative
        # flip for visualization
        if class_name == "opinion":
            expl *= (-1)
        
        
        #convert explanation to human readable format
        tokens = convert_to_tok(input_ids.cpu().flatten(), tokenizer, special_tokens=special_tokens[0])
        tokensFinal, expli = retokenize(tokens, expl)

        
        # Uncomment this line to normalize
        #expli = normalize(expli)
        
        expls_bert.append([(i[0].item(), i[1]) for i in zip(expli, itm.split())])
        
        
        input_ids.cpu()
        attention_mask.cpu()
        model.zero_grad(set_to_none=True)
        del expl
        del output 
        del tokens
        del tokensFinal
        del expli
        del input_ids
        del attention_mask
    
    torch.set_grad_enabled(False)
    gc.collect()
    del model
    del tokenizer
    del explainer
    
    
    return expls_bert

In [ ]:
from Code_helpers.data_processing import preprocess, tokenize, create_dataset
from Code_helpers.model_explainer import get_model, save_model
from transformers.utils import logging

logging.set_verbosity_error()

model = AutoModelForSequenceClassification.from_pretrained('camembert-base', output_loading_info=False)
tokenizer = AutoTokenizer.from_pretrained('camembert-base', do_lowercase=True)

dataDfInit = pd.read_csv("NeededFiles/infopinion_corpus.csv", sep='|')
dataDfInit["text"] = dataDfInit["text"].apply(lambda a : preprocess(a, full=True))
dataDfInit["text"] = dataDfInit["text"].apply(lambda a : get_text_tokens(a,  tokenizer))
dataDfInit.head()

In [ ]:
#This section's sole purpose is to make sure the texts we want to explain were not put in the training dataset
#It can be removed if working with other texts

text_list = ['Tous les deux coupables de l’assassinat d’Emilie Tyberghein',
             'Alep: l’insoutenable impuissance de l’Europe',
             'La «Nouvelle Globalisation»',
             'Il faut sauver Delhaize',
             'Forums des médias en ligne: Les insultes tuent le message ""',
             "Affaire Ronan Farrow: backlash de l'affaire Weinstein ? "] 

unkept = dataDfInit[dataDfInit.title.isin(text_list)]
kept = dataDfInit[~dataDfInit.title.isin(text_list)]

train_df, val_df, test_df = create_dataset(kept, seed = 42)
test_df = pd.concat([test_df, unkept], axis=0).reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))

In [ ]:
train_df = train_df[['text', 'label']]
val_df = val_df[['text', 'label']]
test_df = test_df[['text', 'label']]

train_text =  train_df['text']
val_text = val_df[['text']]
test_text = test_df[['text']]

In [ ]:
#1) Extract the embeddings and merge them to df for frozen model training
seed = 0
set_seed(seed)

train_ds = datasets.Dataset.from_pandas(train_df.sample(frac=1, random_state = seed))
val_ds = datasets.Dataset.from_pandas(val_df.sample(frac=1, random_state = seed))
test_ds = datasets.Dataset.from_pandas(test_df.sample(frac=1, random_state = seed))

rtbf_dataset = datasets.DatasetDict({"train":train_ds,"val":val_ds, "test": test_ds})

tokenized_rtbf_full = rtbf_dataset.map(lambda a: tokenize(a["text"], tokenizer, shorten=True), batched=True)

tokenized_rtbf = tokenized_rtbf_full.remove_columns(['text', '__index_level_0__', 'special_tokens'])

max_length = 512
batch_size = 4
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, max_length = max_length, padding = 'max_length')

g_train = torch.Generator()
g_train.manual_seed(seed)

g_val = torch.Generator()
g_val.manual_seed(seed)

g_test = torch.Generator()
g_test.manual_seed(seed)


train = DataLoader(tokenized_rtbf["train"], 
                   shuffle=False,
                   batch_size=batch_size,
                   collate_fn=data_collator, 
                   worker_init_fn=seed_worker,
                   generator=g_train)

val = DataLoader(tokenized_rtbf["val"], 
                   shuffle=False,
                   batch_size=batch_size,
                   collate_fn=data_collator, 
                   worker_init_fn=seed_worker,
                   generator=g_val)

test = DataLoader(tokenized_rtbf["test"], 
                   shuffle=False,
                   batch_size=batch_size,
                   collate_fn=data_collator,
                   worker_init_fn=seed_worker,
                   generator=g_test)

In [ ]:
text_to_annotate = pd.read_excel('NeededFiles/texts_1to10.xlsx')

title_10 = ["Une Américaine plaide coupable d’avoir tué son petit ami pour des clics sur YouTube",
            "Les Belges, de plus en plus conquis par la Bourse",
            "«Enjeux»: Quand j’entends le mot «liberté»…",
            "Guerre au Proche-Orient : nouvelles frappes sur Beyrouth, des projectiles tirés depuis le Liban vers le centre d'Israë",
            "Le Vacci bus prend ses quartiers à Saint-Josse jusqu'au 30 septembre",
            "Nollet et les mesures «absurdes»: exercer le pouvoir, c’est autre chose que de la tactique et de la séduction",
            "Edito : la concertation sociale à l’agonie",
            "Le voile, ce drapeau de l'islamisme",
            "Après Damso, ne gardons que les Diables moralement irréprochables",
            "Georges-Louis Bouchez s'insurge à propos de l'autorisation du burkini à la piscine Flow à Anderlecht"
           ]

test_text_expl_10 = [preprocess(i, full=True) for i in text_to_annotate[text_to_annotate.title.isin(title_10)]["text"]]
test_text_expl_50 = [preprocess(i, full=True) for i in text_to_annotate[~text_to_annotate.title.isin(title_10)]["text"]]

In [ ]:
test_text_expl = test_text_expl_10

In [ ]:
#This section is only to retrain all the models and to get their explanations on the texts selected previously
#Be carefull as it may take multiple hours/days and use around 200GB of memory. 
#Set to false as default, as the explanations are already available in the NeededFiles directory
import pickle

RETRAIN_MODELS=False
if RETRAIN_MODELS:
    start_memory = get_gpu_memory()[0]

    for seed in range(0,200):
        current_memory = get_gpu_memory()[0]
        current_memory = get_cpu_memory()
        print(f"Memory leak = {current_memory - start_memory}")
        print(f"RUNNING SEED {seed}")
        set_seed(seed)
        model = AutoModelForSequenceClassification.from_pretrained('camembert-base', output_loading_info=False)
        model.to(device)

        #We train on the same examples but in different order
        train_ds = datasets.Dataset.from_pandas(train_df.sample(frac=1, random_state = seed))
        val_ds = datasets.Dataset.from_pandas(val_df.sample(frac=1, random_state = seed))
        test_ds = datasets.Dataset.from_pandas(test_df.sample(frac=1, random_state = seed))


        rtbf_dataset = datasets.DatasetDict({"train":train_ds,"val":val_ds, "test": test_ds})

        tokenized_rtbf_full = rtbf_dataset.map(lambda a: tokenize(a["text"], tokenizer, shorten=True), batched=True)
        tokenized_rtbf = tokenized_rtbf_full.remove_columns(['text', '__index_level_0__', 'special_tokens'])


        max_length = 512
        batch_size = 4
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer, max_length = max_length, padding = 'max_length')


        g_train = torch.Generator()
        g_train.manual_seed(seed)

        g_val = torch.Generator()
        g_val.manual_seed(seed)

        g_test = torch.Generator()
        g_test.manual_seed(seed)


        train = DataLoader(tokenized_rtbf["train"], 
                           shuffle=False,
                           batch_size=batch_size,
                           collate_fn=data_collator, 
                           worker_init_fn=seed_worker,
                           generator=g_train)

        val = DataLoader(tokenized_rtbf["val"], 
                           shuffle=False,
                           batch_size=batch_size,
                           collate_fn=data_collator, 
                           worker_init_fn=seed_worker,
                           generator=g_val)

        test = DataLoader(tokenized_rtbf["test"], 
                           shuffle=False,
                           batch_size=batch_size,
                           collate_fn=data_collator,
                           worker_init_fn=seed_worker,
                           generator=g_test)

        #Train
        trained_model, _ = train_model(model, train, val, device, epochs=2, inplace=True)
        
        #Test
        test_acc = test_model(trained_model, test, device)
        
        #Save to avoid retraining
        save_model(trained_model, tokenizer, f'NeededFiles/models/ft_dec_2024/camembert_ft_seed_{seed}')
        trained_model.cpu()

        #Explaination generation
        model_expl_ft = CamembertForSequenceClassification.from_pretrained(f"NeededFiles/models/ft_dec_2024/camembert_ft_seed_{seed}")
        model_expl_ft.eval()
        explainer_ft = Generator(model_expl_ft)
        tokenizer_expl_ft = AutoTokenizer.from_pretrained(f"NeededFiles/models/ft_dec_2024/camembert_ft_seed_{seed}")
        model_expl_ft.to("cuda")
        dict_explanations_ft = explain_camembert(test_text_expl, tokenizer_expl_ft, model_expl_ft, explainer_ft)
        
        
        #Save for further use
        with open(f'NeededFiles/explanations_LRP.pickle', 'ab') as f:
            pickle.dump(dict_explanations_ft, f)

        with open(f'NeededFiles/model_accuracies', 'a+') as f:
            f.write(f'{test_acc}\n')

        del model_expl_ft
        del dict_explanations_ft
        del trained_model
        del model
        del explainer_ft
        del tokenizer_expl_ft

In [ ]:
#Load expls
import pickle
count = 0
all_expls = []
with open(f'NeededFiles/explanations_LRP.pickle', 'rb') as f:
    while True:   
        try:
            dict_explanations_ft = pickle.load(f)
            all_expls+=[dict_explanations_ft]
            count+=1
        except:
            print(count)
            break

200


In [ ]:
#Load model accuracy:
accuracies = []
with open(f'NeededFiles/model_accuracies', 'r') as f:
    for line in f:
        accuracies += [float(line.replace("\n", ""))]

In [20]:
def compute_epsilon(accuracies, n = 50, best=False):
    sorted_acc = sorted([(i, pos) for pos, i in enumerate(accuracies)])
    if not best:
        closest_epsilon_n = sorted_acc[-1][0] - sorted_acc[-n][0]
        closest_indices = [i[1] for i in sorted_acc[-n:]]
        for starting_pos in range(len(sorted_acc)-n):
            epsilon_n = sorted_acc[starting_pos + n][0] - sorted_acc[starting_pos][0]
            if epsilon_n < closest_epsilon_n:
                closest_epsilon_n = epsilon_n
                closest_indices = [i[1] for i in sorted_acc[starting_pos:starting_pos+n]]
        assert len(closest_indices) == n
        return closest_epsilon_n, closest_indices
    else:
        best_indices = [i[1] for i in sorted_acc[-n:]]
        best_epsilon_n = sorted_acc[-1][0] - sorted_acc[-n][0]
        assert len(best_indices) == n
        return best_epsilon_n, best_indices

In [22]:
epsilon_200_closest, indices_200_closest = compute_epsilon(accuracies, n = 200, best=False)
epsilon_150_closest, indices_150_closest = compute_epsilon(accuracies, n = 150, best=False)
epsilon_100_closest, indices_100_closest = compute_epsilon(accuracies, n = 100, best=False)
epsilon_50_closest, indices_50_closest = compute_epsilon(accuracies, n = 50, best=False)

epsilon_200_best, indices_200_best = compute_epsilon(accuracies, n = 200, best=True)
epsilon_150_best, indices_150_best = compute_epsilon(accuracies, n = 150, best=True)
epsilon_100_best, indices_100_best = compute_epsilon(accuracies, n = 100, best=True)
epsilon_50_best, indices_50_best = compute_epsilon(accuracies, n = 50, best=True)

In [23]:
print(epsilon_200_closest, epsilon_150_closest, epsilon_100_closest, epsilon_50_closest)

0.029999999999999916 0.01200000000000001 0.006000000000000005 0.0020000000000000018


In [24]:
acc_200_closest = [i for pos, i in enumerate(accuracies) if pos in indices_200_closest]
acc_150_closest = [i for pos, i in enumerate(accuracies) if pos in indices_150_closest]
acc_100_closest = [i for pos, i in enumerate(accuracies) if pos in indices_100_closest]
acc_50_closest = [i for pos, i in enumerate(accuracies) if pos in indices_50_closest]

acc_200_best = [i for pos, i in enumerate(accuracies) if pos in indices_200_best]
acc_150_best = [i for pos, i in enumerate(accuracies) if pos in indices_150_best]
acc_100_best = [i for pos, i in enumerate(accuracies) if pos in indices_100_best]
acc_50_best = [i for pos, i in enumerate(accuracies) if pos in indices_50_best]

In [26]:
import math
import scipy.stats as stats

#from https://online.stat.psu.edu/stat415/lesson/9/9.4

def z_score(p1, p2, n):
    P = (p1 + p2)/2
    return (p1-p2)/math.sqrt(P*(1-P)/n)
    
z = z_score(min(acc_200_closest), max(acc_200_closest), 1000)
print(stats.norm.cdf(z))


z = z_score(min(acc_150_closest), max(acc_150_closest), 1000)
print(stats.norm.cdf(z))
z = z_score(min(acc_150_best), max(acc_150_best), 1000)
print(stats.norm.cdf(z))

z = z_score(min(acc_100_closest), max(acc_100_closest), 1000)
print(stats.norm.cdf(z))
z = z_score(min(acc_100_best), max(acc_100_best), 1000)
print(stats.norm.cdf(z))

z = z_score(min(acc_50_closest), max(acc_50_closest), 1000)
print(stats.norm.cdf(z))
z = z_score(min(acc_50_best), max(acc_50_best), 1000)
print(stats.norm.cdf(z))

0.00014498597206234434
0.0735795939860693
0.011878092404488588
0.234281371063235
0.03728269793438564
0.40335400802101823
0.07837454151539593


In [28]:
print(min(acc_200_closest), max(acc_200_closest))

print(min(acc_150_closest), max(acc_150_closest))
print(min(acc_150_best), max(acc_150_best))

print(min(acc_100_closest), max(acc_100_closest))
print(min(acc_100_best), max(acc_100_best))

print(min(acc_50_closest), max(acc_50_closest))
print(min(acc_50_best), max(acc_50_best))

0.911 0.941
0.92 0.932
0.923 0.941
0.923 0.929
0.927 0.941
0.927 0.929
0.93 0.941


In [29]:
import numpy as np
import scipy.special as special
from scipy.stats import pearsonr

def pearsonr_fisher_ci(r, n, confidence_level, alternative):
    """
    Compute the confidence interval for Pearson's R.

    Fisher's transformation is used to compute the confidence interval
    (https://en.wikipedia.org/wiki/Fisher_transformation).
    """
    if r == 1:
        zr = np.inf
    elif r == -1:
        zr = -np.inf
    else:
        zr = np.arctanh(r)

    if n > 3:
        se = np.sqrt(1 / (n - 3))
        if alternative == "two-sided":
            h = special.ndtri(0.5 + confidence_level/2)
            zlo = zr - h*se
            zhi = zr + h*se
            rlo = np.tanh(zlo)
            rhi = np.tanh(zhi)
        elif alternative == "less":
            h = special.ndtri(confidence_level)
            zhi = zr + h*se
            rhi = np.tanh(zhi)
            rlo = -1.0
        else:
            # alternative == "greater":
            h = special.ndtri(confidence_level)
            zlo = zr - h*se
            rlo = np.tanh(zlo)
            rhi = 1.0
    else:
        rlo, rhi = -1.0, 1.0

    return rlo, rhi


def seed_correlation(text_nbr, indices = None, bootstrap = False, latex_format = False):
    if indices==None:
        expls = cp.deepcopy(all_expls)
    else:
        expls = cp.deepcopy([all_expls[i] for i in indices])
    if bootstrap:
        corrs = [0 for _ in range(10000)]
        for j in range(10000):
            random.shuffle(expls)
            seed_nbr = len(expls)//2
            pearson1 = [item[0] for i in expls[:seed_nbr] for item in i[text_nbr]]
            pearson2 = [item[0] for i in expls[-seed_nbr:] for item in i[text_nbr]]
            corr, _ = pearsonr(pearson1, pearson2)
            corrs[j] = corr
        bound = int(0.025*len(corrs))
        ci = sorted(corrs)[bound], sorted(corrs)[-bound]
        correlation = (sum(corrs)/len(corrs))
    else:
        seed_nbr = len(expls)//2
        pearson1 = [item[0] for i in expls[:seed_nbr] for item in i[text_nbr]]
        pearson2 = [item[0] for i in expls[-seed_nbr:] for item in i[text_nbr]]
        print(len(pearson1))
        correlation, _ = pearsonr(pearson1, pearson2)
        ci = pearsonr_fisher_ci(correlation, len(pearson1), 0.95, "two-sided")
        
    if latex_format:
        correlation = format(correlation, '0.4f')
        lb, hb = format(ci[0], '0.4f'), format(ci[1], '0.4f')
        return f'{correlation} & [{lb}; {hb}]'
    else:
        return correlation, ci

In [30]:
import copy as cp
print(seed_correlation(0, indices_200_best))

print(seed_correlation(0, indices_150_closest))
print(seed_correlation(0, indices_150_best))

print(seed_correlation(0, indices_100_closest))
print(seed_correlation(0, indices_100_best))

print(seed_correlation(0, indices_50_closest))
print(seed_correlation(0, indices_50_best))


print(seed_correlation(7, indices_200_best))

print(seed_correlation(7, indices_150_closest))
print(seed_correlation(7, indices_150_best))

print(seed_correlation(7, indices_100_closest))
print(seed_correlation(7, indices_100_best))

print(seed_correlation(7, indices_50_closest))
print(seed_correlation(7, indices_50_best))

9200
(np.float64(0.3102446311455734), (np.float64(0.29165913848336983), np.float64(0.3285959554129249)))
6900
(np.float64(0.3295115453736537), (np.float64(0.3083127750594514), np.float64(0.3503832123168192)))
6900
(np.float64(0.3592756677519825), (np.float64(0.3385497618746297), np.float64(0.3796531220080905)))
4600
(np.float64(0.323406075092851), (np.float64(0.2972851094977646), np.float64(0.34904329499945064)))
4600
(np.float64(0.2946681182896877), (np.float64(0.26805130607947314), np.float64(0.3208354339211142)))
2300
(np.float64(0.2998341504383783), (np.float64(0.2621750856263597), np.float64(0.3365813800393172)))
2300
(np.float64(0.33969507990014025), (np.float64(0.30303038464823645), np.float64(0.3753556111578652)))
14400
(np.float64(0.3590046979358714), (np.float64(0.34469259839883143), np.float64(0.3731499314966694)))
10800
(np.float64(0.36314021110045247), (np.float64(0.34665425805287114), np.float64(0.37940187970815176)))
10800
(np.float64(0.36325487608501683), (np.float64(0.